In [1]:
# ============================================================
# SEQUENTIAL COVERING ALGORITHM
# CAR EVALUATION DATASET
# ============================================================

# Run this cell in Google Colab

!pip install networkx -q


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx


# ============================================================
# 2. LOAD CAR EVALUATION DATASET
# ============================================================

# Download the dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/car/car.data"

columns = [
    "buying",
    "maint",
    "doors",
    "persons",
    "lug_boot",
    "safety",
    "class"
]

df = pd.read_csv(
    url,
    names=columns
)


# ============================================================
# 3. DATASET INFORMATION
# ============================================================

TARGET_ATTR = "class"

# We will learn rules for the "good" class
TARGET_VAL = "good"

ATTRIBUTES = [
    c for c in columns
    if c != TARGET_ATTR
]


# Domain: possible values of every attribute
DOMAIN = {
    attr: sorted(
        df[attr].unique()
    )
    for attr in ATTRIBUTES
}


# ============================================================
# 4. PRINT DATASET
# ============================================================

def print_dataset():

    print("=" * 80)
    print("STEP 0 : CAR EVALUATION DATASET")
    print("=" * 80)

    print(df.head(20).to_string(index=True))

    print("\nDataset shape:", df.shape)

    print("\nClass distribution:")
    print(
        df[TARGET_ATTR].value_counts()
    )

    n_pos = (
        df[TARGET_ATTR] == TARGET_VAL
    ).sum()

    n_neg = (
        df[TARGET_ATTR] != TARGET_VAL
    ).sum()

    print(
        f"\nTarget class: {TARGET_VAL}"
    )

    print(
        f"Positive examples: {n_pos}"
    )

    print(
        f"Negative examples: {n_neg}"
    )

    print()


# ============================================================
# 5. RULE REPRESENTATION
# ============================================================

def covers(rule, row):

    """
    Check whether a row satisfies
    every condition in the rule.
    """

    return all(
        row[attr] == val
        for attr, val in rule.items()
    )


def evaluate(rule, examples):

    """
    Return:

    positives covered
    negatives covered
    precision
    """

    covered = examples[
        examples.apply(
            lambda r: covers(rule, r),
            axis=1
        )
    ]

    pos = (
        covered[TARGET_ATTR]
        == TARGET_VAL
    ).sum()

    neg = (
        covered[TARGET_ATTR]
        != TARGET_VAL
    ).sum()

    precision = (
        pos / (pos + neg)
        if (pos + neg) > 0
        else 0.0
    )

    return pos, neg, precision


def rule_to_str(rule):

    if not rule:
        return "IF (true)"

    conds = " AND ".join(
        f"{a}={v}"
        for a, v in rule.items()
    )

    return f"IF {conds}"


# ============================================================
# 6. LEARN-ONE-RULE
# GENERAL-TO-SPECIFIC BEAM SEARCH
# ============================================================

def learn_one_rule(
    examples,
    beam_width=4,
    max_conditions=None
):

    """
    General-to-specific Beam Search.

    Starts with:

        IF true

    Then adds one attribute=value
    condition at a time.
    """

    if max_conditions is None:
        max_conditions = len(
            ATTRIBUTES
        )

    search_history = []


    # --------------------------------------------------------
    # DEPTH 0
    # --------------------------------------------------------

    beam = [{}]

    depth0_pos, depth0_neg, depth0_prec = (
        evaluate(
            {},
            examples
        )
    )


    search_history.append({

        "depth": 0,

        "candidates": [
            (
                {},
                depth0_pos,
                depth0_neg,
                depth0_prec,
                None
            )
        ],

        "beam": [{}]

    })


    best_rule = {}

    best_pos = depth0_pos
    best_neg = depth0_neg
    best_prec = depth0_prec


    # --------------------------------------------------------
    # SEARCH
    # --------------------------------------------------------

    for depth in range(
        1,
        max_conditions + 1
    ):

        candidates = []


        for parent in beam:

            used_attrs = set(
                parent.keys()
            )


            for attr in ATTRIBUTES:

                if attr in used_attrs:
                    continue


                for val in DOMAIN[attr]:

                    child = dict(parent)

                    child[attr] = val


                    pos, neg, prec = evaluate(
                        child,
                        examples
                    )


                    # Ignore rules covering
                    # no positive examples
                    if pos == 0:
                        continue


                    candidates.append(
                        (
                            child,
                            pos,
                            neg,
                            prec,
                            parent
                        )
                    )


        if not candidates:
            break


        # ----------------------------------------------------
        # RANK CANDIDATES
        # ----------------------------------------------------

        candidates.sort(
            key=lambda c: (
                c[2] == 0,
                c[3],
                c[1]
            ),
            reverse=True
        )


        # ----------------------------------------------------
        # KEEP TOP BEAM WIDTH
        # ----------------------------------------------------

        new_beam = []

        seen = set()


        for (
            child,
            pos,
            neg,
            prec,
            parent
        ) in candidates:

            key = frozenset(
                child.items()
            )


            if key not in seen:

                seen.add(key)

                new_beam.append(
                    child
                )


            if len(new_beam) >= beam_width:
                break


        beam = new_beam


        search_history.append({

            "depth": depth,

            "candidates": candidates,

            "beam": beam

        })


        # ----------------------------------------------------
        # CHECK FOR PURE RULE
        # ----------------------------------------------------

        pure_candidates = [
            c
            for c in candidates
            if c[2] == 0
        ]


        if pure_candidates:

            top = pure_candidates[0]

            (
                best_rule,
                best_pos,
                best_neg,
                best_prec,
                _
            ) = top

            break


        else:

            top = candidates[0]


            if (
                (
                    top[2] == 0,
                    top[3],
                    top[1]
                )
                >
                (
                    best_neg == 0,
                    best_prec,
                    best_pos
                )
            ):

                (
                    best_rule,
                    best_pos,
                    best_neg,
                    best_prec,
                    _
                ) = top


    return (
        best_rule,
        best_pos,
        best_neg,
        best_prec,
        search_history
    )


# ============================================================
# 7. SEQUENTIAL COVERING
# ============================================================

def sequential_covering(
    beam_width=4
):

    working = df.copy()

    rules = []

    round_logs = []

    round_num = 1


    # Continue until all positive
    # examples have been covered

    while (
        working[TARGET_ATTR]
        == TARGET_VAL
    ).sum() > 0:


        (
            rule,
            pos,
            neg,
            prec,
            history
        ) = learn_one_rule(
            working,
            beam_width=beam_width
        )


        # ----------------------------------------------------
        # FIND COVERED EXAMPLES
        # ----------------------------------------------------

        covered_mask = working.apply(
            lambda r: covers(
                rule,
                r
            ),
            axis=1
        )


        covered_examples = working[
            covered_mask
        ]


        # Positive examples covered
        covered_positive_idx = (
            covered_examples[
                covered_examples[TARGET_ATTR]
                == TARGET_VAL
            ].index.tolist()
        )


        # ----------------------------------------------------
        # PRINT ROUND INFORMATION
        # ----------------------------------------------------

        print(
            f"--- Round {round_num} ---"
        )


        print(
            "Remaining positives before this round:",
            (
                working[TARGET_ATTR]
                == TARGET_VAL
            ).sum()
        )


        print(
            "Learned rule:",
            rule_to_str(rule)
        )


        print(
            f"Rule precision: {prec:.4f}"
        )


        print(
            f"Covers: {pos} positive(s), "
            f"{neg} negative(s)"
        )


        print(
            "Positive example indices covered:",
            covered_positive_idx
        )


        print()


        # ----------------------------------------------------
        # SAVE RULE
        # ----------------------------------------------------

        rules.append(rule)


        round_logs.append({

            "round": round_num,

            "rule": rule,

            "pos_covered": pos,

            "neg_covered": neg,

            "precision": prec,

            "covered_positive_idx":
                covered_positive_idx,

            "history": history,

            "remaining_pos_before":
                (
                    working[TARGET_ATTR]
                    == TARGET_VAL
                ).sum()

        })


        # ----------------------------------------------------
        # REMOVE COVERED POSITIVES
        # ----------------------------------------------------

        working = working.drop(
            index=covered_positive_idx
        )


        round_num += 1


        # Safety check
        if pos == 0:

            print(
                "No further progress possible."
            )

            break


    return (
        rules,
        round_logs
    )


# ============================================================
# 8. SEARCH TREE GRAPH
# ============================================================

def plot_search_tree(
    history,
    winning_rule,
    round_num
):

    G = nx.DiGraph()

    labels = {}

    node_colors = []

    node_id_of_rule = {}


    def node_id(rule):

        key = frozenset(
            rule.items()
        )


        if key not in node_id_of_rule:

            node_id_of_rule[key] = (
                f"n{len(node_id_of_rule)}"
            )


        return node_id_of_rule[key]


    # --------------------------------------------------------
    # ROOT
    # --------------------------------------------------------

    root_pos, root_neg, root_prec = (
        history[0]["candidates"][0][1:4]
    )


    root_id = node_id({})


    G.add_node(root_id)


    labels[root_id] = (
        f"(true)\n"
        f"pos={root_pos} "
        f"neg={root_neg}"
    )


    winning_key = frozenset(
        winning_rule.items()
    )


    # --------------------------------------------------------
    # BEAM NODES
    # --------------------------------------------------------

    kept_keys = set()


    for level in history:

        for r in level["beam"]:

            kept_keys.add(
                frozenset(
                    r.items()
                )
            )


    kept_keys.add(
        winning_key
    )


    # --------------------------------------------------------
    # CREATE GRAPH
    # --------------------------------------------------------

    for level in history[1:]:

        for (
            child,
            pos,
            neg,
            prec,
            parent
        ) in level["candidates"]:


            key = frozenset(
                child.items()
            )


            if key not in kept_keys:
                continue


            cid = node_id(child)

            pid = node_id(parent)


            if not G.has_node(cid):

                G.add_node(cid)


                label = " ∧ ".join(
                    f"{a}={v}"
                    for a, v in child.items()
                )


                labels[cid] = (
                    f"{label}\n"
                    f"pos={pos} "
                    f"neg={neg}"
                )


            G.add_edge(
                pid,
                cid
            )


    # --------------------------------------------------------
    # LAYOUT
    # --------------------------------------------------------

    for layer, nodes in enumerate(
        nx.bfs_layers(
            G,
            root_id
        )
    ):

        for n in nodes:

            G.nodes[n]["layer"] = layer


    pos_layout = nx.multipartite_layout(
        G,
        subset_key="layer"
    )


    # --------------------------------------------------------
    # COLORS
    # --------------------------------------------------------

    for n in G.nodes():

        key = None


        for k, v in node_id_of_rule.items():

            if v == n:

                key = k

                break


        if key == winning_key:

            node_colors.append(
                "#02C39A"
            )

        elif n == root_id:

            node_colors.append(
                "#028090"
            )

        else:

            node_colors.append(
                "#CFE8E5"
            )


    # --------------------------------------------------------
    # DRAW
    # --------------------------------------------------------

    plt.figure(
        figsize=(12, 7)
    )


    nx.draw(
        G,
        pos_layout,
        labels=labels,
        with_labels=True,
        node_color=node_colors,
        node_size=3400,
        font_size=7.5,
        font_weight="bold",
        arrows=True,
        edge_color="#8FB9B5",
        linewidths=1.2,
        edgecolors="#0B2E33"
    )


    plt.title(
        f"Round {round_num}: "
        f"General-to-Specific Search Tree\n"
        f"Winning Rule Highlighted",
        fontsize=13,
        fontweight="bold"
    )


    winner_patch = mpatches.Patch(
        color="#02C39A",
        label="Winning rule"
    )


    root_patch = mpatches.Patch(
        color="#028090",
        label="Most general rule"
    )


    other_patch = mpatches.Patch(
        color="#CFE8E5",
        label="Other beam candidates"
    )


    plt.legend(
        handles=[
            winner_patch,
            root_patch,
            other_patch
        ],
        loc="lower center",
        bbox_to_anchor=(0.5, -0.12),
        ncol=3,
        frameon=False
    )


    plt.show()


# ============================================================
# 9. COVERAGE BAR CHART
# ============================================================

def plot_coverage_bar(
    round_logs
):

    rounds = [
        f"Round {rl['round']}"
        for rl in round_logs
    ]


    covered = [
        rl["pos_covered"]
        for rl in round_logs
    ]


    remaining_after = []


    total_pos = (
        df[TARGET_ATTR]
        == TARGET_VAL
    ).sum()


    running = total_pos


    for rl in round_logs:

        running -= rl[
            "pos_covered"
        ]

        remaining_after.append(
            running
        )


    x = range(
        len(rounds)
    )


    fig, ax = plt.subplots(
        figsize=(9, 5)
    )


    ax.bar(
        x,
        covered,
        color="#02C39A",
        label="Positives covered this round"
    )


    ax.bar(
        x,
        remaining_after,
        bottom=covered,
        color="#E8F0EF",
        edgecolor="#028090",
        label="Positives remaining"
    )


    for i, (c, r) in enumerate(
        zip(
            covered,
            remaining_after
        )
    ):

        if c > 0:

            ax.text(
                i,
                c / 2,
                str(c),
                ha="center",
                va="center",
                fontweight="bold",
                color="white"
            )


        if r > 0:

            ax.text(
                i,
                c + r / 2,
                str(r),
                ha="center",
                va="center",
                fontweight="bold"
            )


    ax.set_xticks(
        list(x)
    )


    ax.set_xticklabels(
        rounds
    )


    ax.set_ylabel(
        "Number of positive examples"
    )


    ax.set_title(
        "Sequential Covering: "
        "Positive Examples Covered per Round",
        fontsize=13,
        fontweight="bold"
    )


    ax.legend()

    ax.spines[
        ["top", "right"]
    ].set_visible(False)


    plt.tight_layout()

    plt.show()


# ============================================================
# 10. FINAL COVERAGE HEATMAP
# ============================================================

def plot_coverage_heatmap(
    rules
):

    n = len(df)

    m = len(rules)


    matrix = [

        [

            1 if covers(
                rule,
                df.loc[i]
            )

            else 0

            for rule in rules

        ]

        for i in range(n)
    ]


    fig, ax = plt.subplots(
        figsize=(4.5 + m, 12)
    )


    ax.imshow(
        matrix,
        cmap="Greens",
        aspect="auto",
        vmin=0,
        vmax=1
    )


    ax.set_xticks(
        range(m)
    )


    ax.set_xticklabels(
        [
            f"Rule {i+1}\n"
            f"{rule_to_str(r)}"

            for i, r in enumerate(rules)

        ],
        fontsize=8
    )


    # Show all examples if dataset is manageable
    ax.set_yticks(
        range(n)
    )


    row_labels = [
        f"{i}: {df.loc[i, TARGET_ATTR]}"
        for i in range(n)
    ]


    ax.set_yticklabels(
        row_labels,
        fontsize=6
    )


    # Mark covered examples
    for i in range(n):

        for j in range(m):

            if matrix[i][j]:

                ax.text(
                    j,
                    i,
                    "✓",
                    ha="center",
                    va="center",
                    color="white",
                    fontweight="bold",
                    fontsize=7
                )


    ax.set_title(
        "Which Rule Covers Which Training Example?",
        fontsize=13,
        fontweight="bold"
    )


    ax.set_xlabel(
        "Learned rules"
    )


    ax.set_ylabel(
        "Training example : true class"
    )


    plt.tight_layout()

    plt.show()


# ============================================================
# 11. MAIN PROGRAM
# ============================================================

print_dataset()


# ------------------------------------------------------------
# RUN SEQUENTIAL COVERING
# ------------------------------------------------------------

print("=" * 80)
print("STEP 1 : RUNNING SEQUENTIAL COVERING")
print("=" * 80)


rules, round_logs = sequential_covering(
    beam_width=4
)


# ------------------------------------------------------------
# FINAL RULE SET
# ------------------------------------------------------------

print("=" * 80)
print("FINAL LEARNED RULE SET")
print("=" * 80)


for i, r in enumerate(
    rules,
    start=1
):

    print(
        f"Rule {i}: "
        f"{rule_to_str(r)} "
        f"THEN "
        f"{TARGET_ATTR} = {TARGET_VAL}"
    )


print()


# ------------------------------------------------------------
# GRAPH GENERATION
# ------------------------------------------------------------

print("=" * 80)
print("STEP 2 : GENERATING GRAPHS")
print("=" * 80)


# Search tree for each round
for rl in round_logs:

    plot_search_tree(
        rl["history"],
        rl["rule"],
        rl["round"]
    )


# Coverage bar chart
plot_coverage_bar(
    round_logs
)


# Final coverage heatmap
plot_coverage_heatmap(
    rules
)

Output hidden; open in https://colab.research.google.com to view.